# MLflow tracking -- phase 6

Phase 5 compared two runs in a pandas table that vanishes when the
kernel restarts. This notebook puts the same two runs somewhere durable.

1. point MLflow at the managed tracking server
2. re-run A and B, logging params, metrics and the model
3. query the runs back and compare
4. register the better artifact for phase 7 to promote

The tracking server bills hourly while it exists -- see the teardown note
at the end.

## 1. Setup

`sagemaker-mlflow` is the plugin that lets the open-source MLflow client
authenticate to the managed server with SigV4. The Studio image may not
ship it.

In [ ]:
%pip install -q mlflow sagemaker-mlflow

The tracking URI is the server ARN, not an https URL:

```
terraform -chdir=infra/domain output -raw mlflow_tracking_server_arn
```

In [ ]:
import io
import json

import boto3
import mlflow
import numpy as np
import pandas as pd

REGION = "ca-central-1"
BUCKET = "sagemaker-domain-dev-data-pqkx2l"

# From `terraform -chdir=infra/domain output -raw mlflow_tracking_server_arn`.
TRACKING_ARN = "REPLACE-ME"

mlflow.set_tracking_uri(TRACKING_ARN)
mlflow.set_experiment("bike-sharing")

s3 = boto3.client("s3", region_name=REGION)

print(f"tracking uri: {mlflow.get_tracking_uri()}")
print(f"experiment:   {mlflow.get_experiment_by_name('bike-sharing').experiment_id}")

## 2. Reload the data

Same `featured/` frame and the same time split as phases 4 and 5. The
runs logged below have to be comparable to what phase 5 measured.

In [ ]:
obj = s3.get_object(Bucket=BUCKET, Key="featured/hour.parquet")
df = pd.read_parquet(io.BytesIO(obj["Body"].read()))

TARGET = "cnt"
FEATURES = [c for c in df.columns if c != TARGET]

train, test = df[df.yr == 0], df[df.yr == 1]
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

print(f"train {len(train)} rows (2011)   test {len(test)} rows (2012)")

## 3. Log the two runs

`mlflow.autolog()` would capture most of this, but logging explicitly
shows what actually lands in the tracking server -- and keeps the metric
names matching phase 5's `eval.json`.

In [ ]:
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RUNS = {
    "A-leaf5": {"n_estimators": 100, "min_samples_leaf": 5},
    "B-leaf1": {"n_estimators": 100, "min_samples_leaf": 1},
}

run_ids = {}

for name, params in RUNS.items():
    with mlflow.start_run(run_name=name) as run:
        model = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
        model.fit(X_train, y_train)

        pred = model.predict(X_test)
        metrics = {
            "rmse": float(np.sqrt(mean_squared_error(y_test, pred))),
            "mae": float(mean_absolute_error(y_test, pred)),
            "r2": float(r2_score(y_test, pred)),
        }

        buf = io.BytesIO()
        joblib.dump(model, buf)
        metrics["artifact_mb"] = buf.tell() / 1024 / 1024

        mlflow.log_params(params)
        mlflow.log_param("split", "2011-train/2012-test")
        mlflow.log_metrics(metrics)

        # Ships the model itself to the artifact store, not just numbers.
        mlflow.sklearn.log_model(model, name="model")

        run_ids[name] = run.info.run_id
        print(f"{name}: rmse={metrics['rmse']:.1f} r2={metrics['r2']:.3f} "
              f"size={metrics['artifact_mb']:.1f}MB  run_id={run.info.run_id[:8]}")

## 4. Query the runs back

This is the part a pandas table cannot do: the runs are queryable after
the kernel dies, by anyone with access to the server.

In [ ]:
runs = mlflow.search_runs(
    experiment_names=["bike-sharing"],
    order_by=["metrics.rmse ASC"],
)

cols = [
    "tags.mlflow.runName",
    "params.min_samples_leaf",
    "metrics.rmse",
    "metrics.r2",
    "metrics.artifact_mb",
]

runs[cols].round(3)

## 5. Cross-check against phase 5

The runs above were re-fit, not copied from `eval.json`. If the metrics
match, the phase 5 reproducibility claim holds across kernels and
sessions -- not just twice in one notebook.

In [ ]:
obj = s3.get_object(Bucket=BUCKET, Key="model/eval.json")
phase5 = json.loads(obj["Body"].read())

for name in RUNS:
    logged = runs.loc[runs["tags.mlflow.runName"] == name, "metrics.rmse"].iloc[0]
    recorded = phase5["runs"][name]["metrics"]["rmse"]

    assert np.isclose(logged, recorded), (
        f"{name}: mlflow {logged} != eval.json {recorded}"
    )
    print(f"{name}: {logged:.4f} matches phase 5")

print()
print("mlflow runs reproduce phase 5 exactly")

## 6. Register the deployment candidate

A is 0.9% worse on rmse and 6x smaller. On the serverless endpoint in
phase 8 -- which reloads the model on every cold start -- the size wins.

Registering records that decision. Phase 7 adds the approval gate on
top, and phase 10 is where bob argues for B.

In [ ]:
result = mlflow.register_model(
    model_uri=f"runs:/{run_ids['A-leaf5']}/model",
    name="bike-sharing-rf",
)

print(f"registered {result.name} version {result.version}")
print(f"source run: {run_ids['A-leaf5']}")

## 7. Open the UI

Run comparison is the thing worth seeing in the browser -- select both
runs, then Compare, and the parallel-coordinates view shows the
size/accuracy tradeoff directly.

```
aws sagemaker create-presigned-mlflow-tracking-server-url \
  --tracking-server-name sagemaker-domain-dev-mlflow \
  --region ca-central-1 --query AuthorizedUrl --output text
```

Or from Studio: the MLflow app in the left sidebar.

## Cost

The tracking server bills hourly for as long as it exists, whether or
not anything is logging to it. It is the most expensive idle resource in
this stack.

Stop it between sessions:

```
aws sagemaker stop-mlflow-tracking-server \
  --tracking-server-name sagemaker-domain-dev-mlflow --region ca-central-1
```

Runs and artifacts survive a stop -- the artifact store is the S3 bucket.